# Statistical Analysis and Hypothesis Testing

## Objectives

This notebook will:

- explain the core principles of descriptive statistics and probability;
- apply these principles to completed online retail invoices;
- compare domestic and international transaction behaviour;
- define and test business hypotheses;
- measure statistical significance and practical effect size;
- explain the assumptions and limitations of the selected methods.

The analysis is performed at invoice level rather than product-line level.
This prevents invoices containing many products from receiving more statistical weight than invoices containing only one product.

## Core statistical and probability principles

### Mean

The mean is calculated by adding all observations and dividing by the number of observations. It is useful for financial planning but can be strongly influenced by unusually large transactions.

### Median

The median is the middle observation after values are ordered. It is less affected by extreme values and can provide a better representation of a typical invoice when the distribution is skewed.

### Variance

Variance measures the average squared distance of observations from their mean. A large variance indicates that the observations are widely dispersed.

### Standard deviation

Standard deviation is the square root of variance. It measures dispersion in the original unit of measurement, making it easier to interpret than variance.

### Probability

Probability describes the likelihood of an event and ranges from zero to one. An empirical probability can be estimated by dividing the number of times an event occurs by the total number of observations.

### Probability distributions

A probability distribution describes how values and their probabilities are distributed. Many statistical tests assume approximately normally distributed data or sampling distributions. Retail transaction values are commonly right-skewed because most orders are relatively small while a few orders are exceptionally large.

### Hypothesis testing

Hypothesis testing evaluates evidence about a population using sample data.

- The null hypothesis, H0, represents no difference or no association.
- The alternative hypothesis, H1, represents the proposed difference or association.
- The significance level, alpha, defines the threshold for rejecting H0.
- The p-value measures how unusual the observed result would be if H0 were true.
- A p-value below alpha provides evidence for rejecting H0.
- Effect size measures the practical strength of a difference or association.

Statistical significance does not automatically demonstrate business importance or causation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

completed_sales_path = (
    project_root
    / "data"
    / "processed"
    / "completed_sales.parquet"
)

assert completed_sales_path.exists()

sales_df = pd.read_parquet(completed_sales_path)

print(f"Completed-sales rows loaded: {len(sales_df):,}")

Completed-sales rows loaded: 524,878


In [3]:
invoice_analysis = (
    sales_df.groupby("InvoiceNo", as_index=False)
    .agg(
        InvoiceRevenue=("LineRevenue", "sum"),
        InvoiceDate=("InvoiceDate", "min"),
        Country=("Country", "first"),
        Units=("Quantity", "sum"),
        ProductLines=("StockCode", "count"),
    )
)

invoice_analysis["Market"] = np.where(
    invoice_analysis["Country"].eq("United Kingdom"),
    "United Kingdom",
    "International",
)

invoice_analysis["IsWeekend"] = (
    invoice_analysis["InvoiceDate"].dt.dayofweek >= 5
)

invoice_analysis.head()

,InvoiceNo,InvoiceRevenue,InvoiceDate,Country,Units,ProductLines,Market,IsWeekend
0,536365,139.12,2010-12-01 08:26:00,United Kingdom,40,7,United Kingdom,False
1,536366,22.20,2010-12-01 08:28:00,United Kingdom,12,2,United Kingdom,False
2,536367,278.73,2010-12-01 08:34:00,United Kingdom,83,12,United Kingdom,False
3,536368,70.05,2010-12-01 08:34:00,United Kingdom,15,4,United Kingdom,False
4,536369,17.85,2010-12-01 08:35:00,United Kingdom,3,1,United Kingdom,False


In [4]:
invoice_statistics = (
    invoice_analysis.groupby("Market")["InvoiceRevenue"]
    .agg(
        InvoiceCount="count",
        Mean="mean",
        Median="median",
        StandardDeviation="std",
        Variance="var",
        Minimum="min",
        Maximum="max",
    )
)

invoice_statistics

,InvoiceCount,Mean,Median,StandardDeviation,Variance,Minimum,Maximum
Market,,,,,,,
International,1941,845.11,424.06,"1,739.82","3,026,989.45",1.45,"22,775.93"
United Kingdom,18019,499.57,299.95,"1,781.52","3,173,815.84",0.38,"168,469.60"


In [5]:
probability_summary = pd.Series(
    {
        "Probability an invoice is international": (
            invoice_analysis["Market"]
            .eq("International")
            .mean()
        ),
        "Probability an invoice occurs at a weekend": (
            invoice_analysis["IsWeekend"].mean()
        ),
        "Probability an invoice exceeds £1,000": (
            invoice_analysis["InvoiceRevenue"]
            .gt(1_000)
            .mean()
        ),
    }
)

probability_summary

Probability an invoice is international      0.10
Probability an invoice occurs at a weekend   0.11
Probability an invoice exceeds £1,000        0.09
dtype: float64

### Probability interpretation

The empirical probability that a completed invoice is international is approximately 9.72%. Most completed invoices therefore originate from the
United Kingdom, which agrees with the geographic revenue analysis.

Approximately 11.04% of completed invoices occurred on a weekend, while about 9.18% exceeded £1,000 in value.

These are empirical probabilities calculated from the supplied historical dataset. They describe the observed period and should not be treated as
guaranteed probabilities for future transactions.

In [6]:
invoice_chart_limit = invoice_analysis[
    "InvoiceRevenue"
].quantile(0.99)

invoice_market_chart = invoice_analysis.loc[
    invoice_analysis["InvoiceRevenue"] <= invoice_chart_limit
].copy()

fig = px.box(
    invoice_market_chart,
    x="Market",
    y="InvoiceRevenue",
    color="Market",
    points=False,
    title=(
        "Invoice Value Distribution by Market "
        "(up to the 99th Percentile)"
    ),
    labels={
        "Market": "Market",
        "InvoiceRevenue": "Invoice value (£)",
    },
)

fig.update_layout(
    showlegend=False,
)

fig.update_yaxes(
    tickprefix="£",
    tickformat=",.0f",
)

fig.show()

### Distribution interpretation

International invoices have a higher observed mean and median value than United Kingdom invoices. Both groups have wide, right-skewed distributions containing large invoice-value outliers.

The chart is restricted to the 99th percentile for readability. No observations were removed from the statistical tables or the underlying dataset.

The visual difference does not by itself demonstrate statistical significance. A formal hypothesis test will be used in the next section.

In [7]:
assert len(invoice_analysis) == sales_df["InvoiceNo"].nunique()
assert invoice_analysis["InvoiceRevenue"].gt(0).all()

assert (
    invoice_statistics["InvoiceCount"].sum()
    == len(invoice_analysis)
)

assert probability_summary.between(0, 1).all()

assert set(invoice_analysis["Market"]) == {
    "United Kingdom",
    "International",
}

print("Statistical foundation calculations validated successfully.")

Statistical foundation calculations validated successfully.


## Hypothesis 1: Domestic and international invoice values

### Business rationale

The geographic analysis found that international markets generate a relatively small proportion of invoices but include several markets with high average invoice values. Testing the difference can help determine whether international orders may require a distinct marketing or account-management strategy.

### Hypotheses

- **H0:** The mean log-transformed invoice value is equal for United Kingdom and international invoices.
- **H1:** The mean log-transformed invoice value differs between United Kingdom and international invoices.

A significance level of **alpha = 0.05** will be used.

A Welch independent-samples t-test is selected because the groups have different sample sizes and equal population variances cannot be assumed. Invoice values are transformed using `log1p` to reduce the influence of the strongly right-skewed distribution.

The test assumes that observations are independent. This assumption may not be completely satisfied because the same customer can place multiple invoices. This limitation will be considered when interpreting the result.

In [8]:
alpha = 0.05

uk_invoice_values = invoice_analysis.loc[
    invoice_analysis["Market"].eq("United Kingdom"),
    "InvoiceRevenue",
]

international_invoice_values = invoice_analysis.loc[
    invoice_analysis["Market"].eq("International"),
    "InvoiceRevenue",
]

uk_log_values = np.log1p(uk_invoice_values)
international_log_values = np.log1p(
    international_invoice_values
)

group_comparison = pd.DataFrame(
    {
        "Market": [
            "United Kingdom",
            "International",
        ],
        "InvoiceCount": [
            len(uk_invoice_values),
            len(international_invoice_values),
        ],
        "MeanInvoiceValue": [
            uk_invoice_values.mean(),
            international_invoice_values.mean(),
        ],
        "MedianInvoiceValue": [
            uk_invoice_values.median(),
            international_invoice_values.median(),
        ],
    }
)

group_comparison

,Market,InvoiceCount,MeanInvoiceValue,MedianInvoiceValue
0,United Kingdom,18019,499.57,299.95
1,International,1941,845.11,424.06


In [9]:
t_statistic, p_value = stats.ttest_ind(
    international_log_values,
    uk_log_values,
    equal_var=False,
)

pooled_standard_deviation = np.sqrt(
    (
        (len(international_log_values) - 1)
        * international_log_values.var(ddof=1)
        + (len(uk_log_values) - 1)
        * uk_log_values.var(ddof=1)
    )
    / (
        len(international_log_values)
        + len(uk_log_values)
        - 2
    )
)

cohens_d = (
    international_log_values.mean()
    - uk_log_values.mean()
) / pooled_standard_deviation

hypothesis_1_result = pd.Series(
    {
        "Significance level": alpha,
        "T-statistic": t_statistic,
        "P-value": p_value,
        "Cohen's d": cohens_d,
        "Decision": (
            "Reject H0"
            if p_value < alpha
            else "Fail to reject H0"
        ),
    }
)

hypothesis_1_result

Significance level         0.05
T-statistic               18.85
P-value                    0.00
Cohen's d                  0.43
Decision              Reject H0
dtype: object

In [10]:
hypothesis_1_chart = invoice_analysis[
    [
        "Market",
        "InvoiceRevenue",
    ]
].copy()

hypothesis_1_chart["LogInvoiceRevenue"] = np.log1p(
    hypothesis_1_chart["InvoiceRevenue"]
)

fig = px.histogram(
    hypothesis_1_chart,
    x="LogInvoiceRevenue",
    color="Market",
    histnorm="probability density",
    barmode="overlay",
    opacity=0.60,
    nbins=60,
    title="Log-Transformed Invoice Values by Market",
    labels={
        "LogInvoiceRevenue": (
            "Log-transformed invoice value"
        ),
        "Market": "Market",
    },
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text="",
)

fig.show()

### Hypothesis 1 conclusion

The p-value is considerably smaller than the 0.05 significance level. Therefore, the null hypothesis is rejected.

The analysis provides strong statistical evidence that log-transformed invoice values differ between the United Kingdom and international markets.
International invoices have higher observed mean and median values.

Cohen's d is approximately 0.435, indicating a small-to-moderate standardised difference. The result is therefore not only statistically significant, but also potentially relevant to international marketing and account-management decisions.

However, the result does not demonstrate that international location causes higher invoice values. International invoices may contain more wholesale customers, bulk orders, or different product combinations.

## Hypothesis 2: Product price and units sold

### Business rationale

Understanding the relationship between product price and sales volume can support pricing and promotional decisions.

### Hypotheses

- **H0:** There is no negative monotonic association between realised average unit price and units sold.
- **H1:** There is a negative monotonic association between realised average unit price and units sold.

A significance level of **alpha = 0.05** will be used.

Spearman's rank correlation is selected because product prices and unit sales are strongly skewed and the relationship may not be linear. Unlike Pearson correlation, Spearman correlation does not require normally distributed variables or a linear relationship.

In [11]:
non_merchandise_codes = {
    "DOT",
    "POST",
    "M",
    "AMAZONFEE",
    "BANK CHARGES",
    "CRUK",
    "D",
    "S",
    "B",
    "PADS",
    "C2",
}

merchandise_sales = sales_df.loc[
    ~sales_df["StockCode"]
    .str.upper()
    .isin(non_merchandise_codes)
].copy()

primary_descriptions = (
    merchandise_sales.groupby(
        ["StockCode", "Description"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "DescriptionFrequency"})
    .sort_values(
        ["StockCode", "DescriptionFrequency"],
        ascending=[True, False],
    )
    .drop_duplicates(subset="StockCode")
    [
        [
            "StockCode",
            "Description",
        ]
    ]
)

product_pricing_analysis = (
    merchandise_sales.groupby(
        "StockCode",
        as_index=False,
    )
    .agg(
        Revenue=("LineRevenue", "sum"),
        UnitsSold=("Quantity", "sum"),
        InvoiceReach=("InvoiceNo", "nunique"),
    )
    .merge(
        primary_descriptions,
        on="StockCode",
        how="left",
        validate="one_to_one",
    )
)

product_pricing_analysis["RealisedAverageUnitPrice"] = (
    product_pricing_analysis["Revenue"]
    / product_pricing_analysis["UnitsSold"]
)

product_pricing_analysis.head()

,StockCode,Revenue,UnitsSold,InvoiceReach,Description,RealisedAverageUnitPrice
0,10002,759.89,860,71,INFLATABLE POLITICAL GLOBE,0.88
1,10080,119.09,303,22,GROOVY CACTUS INFLATABLE,0.39
2,10120,40.32,192,29,DOGGY RUBBER,0.21
3,10123C,3.25,5,3,HEARTS WRAPPING TAPE,0.65
4,10124A,6.72,16,5,SPOTS ON RED BOOKCOVER TAPE,0.42


In [12]:
spearman_correlation, spearman_p_value = (
    stats.spearmanr(
        product_pricing_analysis[
            "RealisedAverageUnitPrice"
        ],
        product_pricing_analysis["UnitsSold"],
        alternative="less",
    )
)

hypothesis_2_result = pd.Series(
    {
        "Significance level": alpha,
        "Spearman correlation": spearman_correlation,
        "P-value": spearman_p_value,
        "Decision": (
            "Reject H0"
            if spearman_p_value < alpha
            else "Fail to reject H0"
        ),
    }
)

hypothesis_2_result

Significance level           0.05
Spearman correlation        -0.38
P-value                      0.00
Decision                Reject H0
dtype: object

In [13]:
fig = px.scatter(
    product_pricing_analysis,
    x="RealisedAverageUnitPrice",
    y="UnitsSold",
    size="InvoiceReach",
    color="Revenue",
    log_x=True,
    log_y=True,
    color_continuous_scale="Viridis",
    title=(
        "Realised Average Unit Price and Product Sales Volume"
    ),
    labels={
        "RealisedAverageUnitPrice": (
            "Realised average unit price (£, logarithmic)"
        ),
        "UnitsSold": "Units sold (logarithmic)",
        "Revenue": "Revenue (£)",
        "InvoiceReach": "Invoice reach",
    },
    hover_name="Description",
    hover_data={
        "StockCode": True,
        "Revenue": ":£,.2f",
        "InvoiceReach": ":,",
    },
)

fig.update_layout(
    coloraxis_colorbar_title="Revenue (£)",
)

fig.show()

### Hypothesis 2 conclusion

The Spearman correlation is approximately -0.379 and the p-value is below the 0.05 significance level. The null hypothesis is therefore rejected.

The result indicates a statistically significant, moderate negative association between realised average unit price and units sold. Lower-priced products tend to sell more units, while higher-priced products tend to sell fewer units.

This association does not prove price elasticity or establish that reducing a price will increase demand. Product type, customer preference, seasonality, wholesale orders, promotions, and product availability may influence both price and sales volume. A controlled pricing experiment would be required to estimate a causal price effect.

In [14]:
assert len(uk_invoice_values) == 18_019
assert len(international_invoice_values) == 1_941

assert np.isfinite(t_statistic)
assert 0 <= p_value <= 1
assert np.isfinite(cohens_d)

assert product_pricing_analysis[
    "RealisedAverageUnitPrice"
].gt(0).all()

assert product_pricing_analysis[
    "UnitsSold"
].gt(0).all()

assert -1 <= spearman_correlation <= 1
assert 0 <= spearman_p_value <= 1

print("Both hypothesis tests validated successfully.")

Both hypothesis tests validated successfully.
